In [1]:
!pip install -q gradio numpy pandas matplotlib

import gradio as gr
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Light sources around Kozan / Adana region (name, lat, lon, light_intensity_index)
LIGHT_SOURCES = [
    {"name": "Kozan Center", "lat": 37.455, "lon": 35.815, "strength": 1.35},
    {"name": "Imamoglu / Kozan South Corridor", "lat": 37.258, "lon": 35.662, "strength": 1.15},
    {"name": "Adana Metro Urban Core", "lat": 37.000, "lon": 35.321, "strength": 4.20},
    {"name": "Ceyhan Industrial / Urban Node", "lat": 37.026, "lon": 35.817, "strength": 2.10},
    {"name": "Kadirli / Osmaniye Corridor", "lat": 37.372, "lon": 36.094, "strength": 1.75},
    {"name": "Feke / Saimbeyli Uplands", "lat": 37.818, "lon": 35.915, "strength": 0.45},
]

CANDIDATE_SITES = [
    {"site": "Kozan Town Outskirts (Lowland)", "lat": 37.445, "lon": 35.845, "elev_m": 220},
    {"site": "Kozan Northern Ridge / Highland", "lat": 37.535, "lon": 35.795, "elev_m": 680},
    {"site": "Tufanbeyli / Toros Foothill Dark Spot", "lat": 37.690, "lon": 35.870, "elev_m": 1150},
    {"site": "Southern Plain Near Highway Corridor", "lat": 37.310, "lon": 35.710, "elev_m": 110},
]


def haversine_km(lat1, lon1, lat2, lon2):
  r = 6371.0
  p1, p2 = np.radians(lat1), np.radians(lat2)
  dp = np.radians(lat2 - lat1)
  dl = np.radians(lon2 - lon1)
  a = np.sin(dp / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dl / 2) ** 2
  return 2 * r * np.arcsin(np.sqrt(np.clip(a, 0.0, 1.0)))


def bearing_deg(lat1, lon1, lat2, lon2):
  p1, p2 = np.radians(lat1), np.radians(lat2)
  dl = np.radians(lon2 - lon1)
  x = np.sin(dl) * np.cos(p2)
  y = np.cos(p1) * np.sin(p2) - np.sin(p1) * np.cos(p2) * np.cos(dl)
  return (np.degrees(np.arctan2(x, y)) + 360.0) % 360.0


def sky_glow_at(lat, lon, elev_m=200.0):
  # Empirical radial light-pollution decay + slight elevation clarity bonus
  total_glow = 0.25
  for src in LIGHT_SOURCES:
    d_km = np.maximum(haversine_km(lat, lon, src["lat"], src["lon"]), 1.5)
    total_glow += src["strength"] * 42.0 / (d_km**1.35)
  elev_factor = np.clip(1.0 - (elev_m - 150.0) / 3500.0, 0.80, 1.05)
  glow = total_glow * elev_factor
  # Map sky glow index to approximate SQM (mag/arcsec^2) and Bortle class (1-9)
  sqm = np.clip(21.95 - 1.15 * np.log1p(glow), 17.5, 22.0)
  bortle = int(np.clip(np.round(9.2 - (sqm - 17.5) * 1.65), 1, 9))
  return float(glow), float(sqm), bortle


def build_observation_plan(
    site_name, target_type, moon_phase, max_cloud_pct, min_alt_deg
):
  site_row = next(
      (s for s in CANDIDATE_SITES if s["site"] == site_name),
      CANDIDATE_SITES[0],
  )
  glow, sqm, bortle = sky_glow_at(
      site_row["lat"], site_row["lon"], site_row["elev_m"]
  )

  # Compute directional sky-dome pollution around the selected site (0..360 deg)
  azimuths = np.linspace(0, 360, 72, endpoint=False)
  dir_pollution = np.zeros_like(azimuths)
  for src in LIGHT_SOURCES:
    d_km = max(
        haversine_km(site_row["lat"], site_row["lon"], src["lat"], src["lon"]),
        1.5,
    )
    b_deg = bearing_deg(
        site_row["lat"], site_row["lon"], src["lat"], src["lon"]
    )
    ang_diff = np.degrees(
        np.arctan2(
            np.sin(np.radians(azimuths - b_deg)),
            np.cos(np.radians(azimuths - b_deg)),
        )
    )
    cone_weight = np.exp(-0.5 * (ang_diff / 28.0) ** 2)
    dir_pollution += (src["strength"] * 35.0 / (d_km**1.25)) * cone_weight

  best_az_idx = int(np.argmin(dir_pollution))
  worst_az_idx = int(np.argmax(dir_pollution))
  best_az = float(azimuths[best_az_idx])
  worst_az = float(azimuths[worst_az_idx])

  # Limiting visual/imaging magnitude estimate
  moon_penalty = (moon_phase / 100.0) * 1.8
  eff_sqm = sqm - moon_penalty
  lim_mag = np.clip((eff_sqm - 16.5) * 1.15 + 1.2, 2.0, 6.8)

  # Target suitability check
  target_bortle_cap = {
      "Bright Planets & Moon": 8,
      "Open & Globular Star Clusters": 6,
      "Faint Nebulae & Galaxies (DSO)": 4,
      "Visible Satellite Passes": 7,
  }
  max_allowed = target_bortle_cap.get(target_type, 6)
  suitability = (
      "HIGHLY SUITABLE"
      if bortle <= max_allowed - 1 and moon_phase < 65
      else (
          "MODERATE / CONDITIONAL"
          if bortle <= max_allowed
          else "CHALLENGING — PICK DARKER SITE"
      )
  )

  # Generate 2D map grid around Kozan / Adana
  lat_grid = np.linspace(36.90, 37.90, 90)
  lon_grid = np.linspace(35.20, 36.20, 90)
  LON_M, LAT_M = np.meshgrid(lon_grid, lat_grid)
  _, SQM_M, _ = sky_glow_at(LAT_M, LON_M, elev_m=400.0)

  plt.style.use("dark_background")
  fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13.5, 5.2), dpi=110)
  fig.patch.set_facecolor("#0b0f17")

  # Panel 1: Local Light Pollution Map (SQM)
  cf = ax1.contourf(
      LON_M, LAT_M, SQM_M, levels=18, cmap="inferno", alpha=0.92
  )
  cbar = fig.colorbar(cf, ax=ax1, pad=0.02)
  cbar.set_label(
      "Estimated Sky Brightness (mag/arcsec²)", color="#cbd5e1", fontsize=9
  )
  cbar.ax.tick_params(colors="#cbd5e1", labelsize=8)

  for s in CANDIDATE_SITES:
    is_sel = s["site"] == site_row["site"]
    ax1.scatter(
        s["lon"],
        s["lat"],
        s=130 if is_sel else 65,
        c="#38bdf8" if is_sel else "#e2e8f0",
        edgecolors="#0b0f17",
        linewidths=1.5,
        zorder=5,
    )
    ax1.text(
        s["lon"] + 0.015,
        s["lat"] + 0.012,
        s["site"].split("(")[0].strip(),
        color="#f8fafc",
        fontsize=8,
        fontweight="bold" if is_sel else "normal",
        zorder=6,
    )

  ax1.set_title(
      "Local Light Pollution Map (Kozan & Surroundings)",
      color="#f8fafc",
      fontsize=11,
      fontweight="bold",
  )
  ax1.set_xlabel("Longitude (°E)", color="#94a3b8", fontsize=9)
  ax1.set_ylabel("Latitude (°N)", color="#94a3b8", fontsize=9)
  ax1.tick_params(colors="#94a3b8", labelsize=8)
  ax1.set_facecolor("#111724")

  # Panel 2: Azimuthal Light-Dome Profile at Selected Site
  ax2.plot(
      azimuths, dir_pollution, color="#38bdf8", linewidth=2.2, label="Light Dome"
  )
  ax2.fill_between(
      azimuths, 0, dir_pollution, color="#38bdf8", alpha=0.18
  )
  ax2.axvline(
      best_az,
      color="#34d399",
      linestyle="--",
      linewidth=1.4,
      label=f"Darkest Azimuth ({best_az:.0f}°)",
  )
  ax2.axvline(
      worst_az,
      color="#f87171",
      linestyle=":",
      linewidth=1.4,
      label=f"Brightest Glare ({worst_az:.0f}°)",
  )
  ax2.set_title(
      f"Directional Light Dome — {site_row['site']}",
      color="#f8fafc",
      fontsize=11,
      fontweight="bold",
  )
  ax2.set_xlabel("Azimuth Direction (0°=N, 90°=E, 180°=S, 270°=W)", color="#94a3b8", fontsize=9)
  ax2.set_ylabel("Relative Directional Sky Glow", color="#94a3b8", fontsize=9)
  ax2.set_xlim(0, 360)
  ax2.tick_params(colors="#94a3b8", labelsize=8)
  ax2.grid(True, alpha=0.18, color="#64748b")
  ax2.legend(
      loc="upper right",
      fontsize=8,
      facecolor="#121826",
      edgecolor="#263248",
  )
  ax2.set_facecolor("#111724")

  plt.tight_layout(pad=1.5)

  cards_html = f"""
    <div style="width:100%;box-sizing:border-box;font-family:Inter,system-ui,sans-serif;color:#e2e8f0;">
      <div style="display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:10px;background:#121826;border:1px solid #263248;border-radius:12px;padding:14px 18px;margin-bottom:12px;">
        <div>
          <div style="font-size:11px;color:#38bdf8;font-weight:700;letter-spacing:0.5px;">ASTRONOMY OBSERVATION PLAN • {site_row['elev_m']} m ELEVATION</div>
          <div style="font-size:20px;font-weight:700;color:#f8fafc;margin-top:2px;">{site_row['site']} <span style="font-size:13px;color:#94a3b8;">({site_row['lat']:.3f}°N, {site_row['lon']:.3f}°E)</span></div>
        </div>
        <span style="background:#0f293a;color:#38bdf8;border:1px solid #0284c7;padding:6px 12px;border-radius:999px;font-size:12px;font-weight:700;">Bortle {bortle} • {suitability}</span>
      </div>

      <div style="display:grid;grid-template-columns:repeat(4,minmax(0,1fr));gap:10px;">
        <div style="background:#131a29;border:1px solid #243044;border-radius:10px;padding:12px;">
          <div style="font-size:11px;color:#94a3b8;">BORTLE CLASS</div>
          <div style="font-size:19px;font-weight:700;color:#38bdf8;margin-top:4px;">Class {bortle} / 9</div>
        </div>
        <div style="background:#131a29;border:1px solid #243044;border-radius:10px;padding:12px;">
          <div style="font-size:11px;color:#94a3b8;">ZENITH SQM</div>
          <div style="font-size:19px;font-weight:700;color:#a78bfa;margin-top:4px;">{sqm:.2f} mag/arcsec²</div>
        </div>
        <div style="background:#131a29;border:1px solid #243044;border-radius:10px;padding:12px;">
          <div style="font-size:11px;color:#94a3b8;">BEST AZIMUTH</div>
          <div style="font-size:19px;font-weight:700;color:#34d399;margin-top:4px;">{best_az:.0f}°</div>
        </div>
        <div style="background:#131a29;border:1px solid #243044;border-radius:10px;padding:12px;">
          <div style="font-size:11px;color:#94a3b8;">EST. LIMITING MAG</div>
          <div style="font-size:19px;font-weight:700;color:#fbbf24;margin-top:4px;">{lim_mag:.1f} mag</div>
        </div>
      </div>

      <div style="background:#111724;border:1px solid #263248;border-radius:12px;padding:15px 18px;margin-top:12px;line-height:1.55;font-size:14px;color:#cbd5e1;">
        <div style="font-weight:700;color:#f8fafc;margin-bottom:6px;">🔭 Actionable Observation Plan — {target_type}</div>
        <ul style="margin:0;padding-left:18px;">
          <li><b>Aim Direction:</b> Favor azimuths near <b>{best_az:.0f}°</b> and keep targets above <b>{min_alt_deg:.0f}°</b> altitude; avoid low-elevation pointing toward <b>{worst_az:.0f}°</b> where urban light domes scatter sky background.</li>
          <li><b>Moon & Sky Window:</b> Current Moon illumination is <b>{moon_phase:.0f}%</b> (cloud cap <b>≤ {max_cloud_pct:.0f}%</b>). For deep-sky targets, prioritize moonless nights (&lt; 25% illumination) after astronomical twilight.</li>
          <li><b>Site Strategy:</b> Moving north/upslope away from urban cores reduces lower-atmosphere scattering and improves contrast for faint objects.</li>
        </ul>
      </div>
    </div>
    """

  schedule_df = pd.DataFrame([
      {
          "Session Window (Local)": "20:00 – 21:30 (Evening Twilight)",
          "Recommended Focus": "Moon, Planets, Bright Clusters & LEO Passes",
          "Min Elevation": f"{max(min_alt_deg, 25):.0f}°",
          "Notes": "Quick setup; early sky gradient near urban corridors",
      },
      {
          "Session Window (Local)": "21:30 – 01:30 (Prime Dark Window)",
          "Recommended Focus": target_type,
          "Min Elevation": f"{min_alt_deg:.0f}°",
          "Notes": f"Aim near {best_az:.0f}° az; check cloud cover <= {max_cloud_pct:.0f}%",
      },
      {
          "Session Window (Local)": "01:30 – 04:30 (Late Night / Pre-Dawn)",
          "Recommended Focus": "Deep-Sky & Low-Glare Meridian Targets",
          "Min Elevation": f"{max(min_alt_deg, 30):.0f}°",
          "Notes": "Reduced street/commercial lighting activity in nearby towns",
      },
  ])

  return cards_html, fig, schedule_df


full_width_css = """
html, body, .gradio-container, .gradio-container > .main, .gradio-container > .main > .wrap, .block, .form {
  background-color: #0b0f17 !important;
  color: #e2e8f0 !important;
  max-width: 100% !important;
  width: 100% !important;
  margin: 0 !important;
  box-sizing: border-box !important;
}
.gradio-container {
  padding: 10px 14px !important;
}
"""

with gr.Blocks() as demo:
  gr.Markdown("## 🌌 Local Light Pollution & Astronomy Observation Planner")

  with gr.Row(equal_height=False):
    with gr.Column(scale=4, min_width=320):
      site_dd = gr.Dropdown(
          choices=[s["site"] for s in CANDIDATE_SITES],
          value=CANDIDATE_SITES[1]["site"],
          label="Candidate Observation Site",
      )
      target_dd = gr.Dropdown(
          choices=[
              "Bright Planets & Moon",
              "Open & Globular Star Clusters",
              "Faint Nebulae & Galaxies (DSO)",
              "Visible Satellite Passes",
          ],
          value="Open & Globular Star Clusters",
          label="Observation Target Type",
      )
      moon_sl = gr.Slider(
          minimum=0,
          maximum=100,
          value=15,
          step=1,
          label="Moon Illumination (%)",
      )
      cloud_sl = gr.Slider(
          minimum=0,
          maximum=100,
          value=20,
          step=5,
          label="Max Acceptable Cloud Cover (%)",
      )
      alt_sl = gr.Slider(
          minimum=10,
          maximum=75,
          value=25,
          step=5,
          label="Minimum Target Elevation (deg)",
      )
      plan_btn = gr.Button(
          "⚡ Map Light Pollution & Build Observation Plan", variant="primary"
      )

    with gr.Column(scale=8, min_width=460):
      out_cards = gr.HTML()
      out_plot = gr.Plot(label="Light Pollution Map & Directional Light Dome")
      out_table = gr.Dataframe(
          label="Nightly Observation Schedule", interactive=False
      )

  plan_btn.click(
      fn=build_observation_plan,
      inputs=[site_dd, target_dd, moon_sl, cloud_sl, alt_sl],
      outputs=[out_cards, out_plot, out_table],
  )

demo.launch(share=False, theme=gr.themes.Soft(), css=full_width_css)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

In [2]:
!pip install -q gradio numpy pandas matplotlib

import gradio as gr
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Light sources around Kozan / Adana region (name, lat, lon, light_intensity_index)
LIGHT_SOURCES = [
    {"name": "Kozan Center", "lat": 37.455, "lon": 35.815, "strength": 1.35},
    {"name": "Imamoglu / Kozan South Corridor", "lat": 37.258, "lon": 35.662, "strength": 1.15},
    {"name": "Adana Metro Urban Core", "lat": 37.000, "lon": 35.321, "strength": 4.20},
    {"name": "Ceyhan Industrial / Urban Node", "lat": 37.026, "lon": 35.817, "strength": 2.10},
    {"name": "Kadirli / Osmaniye Corridor", "lat": 37.372, "lon": 36.094, "strength": 1.75},
    {"name": "Feke / Saimbeyli Uplands", "lat": 37.818, "lon": 35.915, "strength": 0.45},
]

CANDIDATE_SITES = [
    {"site": "Kozan Town Outskirts (Lowland)", "lat": 37.445, "lon": 35.845, "elev_m": 220},
    {"site": "Kozan Northern Ridge / Highland", "lat": 37.535, "lon": 35.795, "elev_m": 680},
    {"site": "Tufanbeyli / Toros Foothill Dark Spot", "lat": 37.690, "lon": 35.870, "elev_m": 1150},
    {"site": "Southern Plain Near Highway Corridor", "lat": 37.310, "lon": 35.710, "elev_m": 110},
]


def haversine_km(lat1, lon1, lat2, lon2):
  r = 6371.0
  p1, p2 = np.radians(lat1), np.radians(lat2)
  dp = np.radians(lat2 - lat1)
  dl = np.radians(lon2 - lon1)
  a = np.sin(dp / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dl / 2) ** 2
  return 2 * r * np.arcsin(np.sqrt(np.clip(a, 0.0, 1.0)))


def bearing_deg(lat1, lon1, lat2, lon2):
  p1, p2 = np.radians(lat1), np.radians(lat2)
  dl = np.radians(lon2 - lon1)
  x = np.sin(dl) * np.cos(p2)
  y = np.cos(p1) * np.sin(p2) - np.sin(p1) * np.cos(p2) * np.cos(dl)
  return (np.degrees(np.arctan2(x, y)) + 360.0) % 360.0


def sky_glow_at(lat, lon, elev_m=200.0):
  total_glow = np.zeros_like(np.asarray(lat, dtype=float)) + 0.25
  for src in LIGHT_SOURCES:
    d_km = np.maximum(haversine_km(lat, lon, src["lat"], src["lon"]), 1.5)
    total_glow = total_glow + (src["strength"] * 42.0 / (d_km**1.35))
  elev_factor = np.clip(1.0 - (np.asarray(elev_m, dtype=float) - 150.0) / 3500.0, 0.80, 1.05)
  glow = total_glow * elev_factor
  sqm = np.clip(21.95 - 1.15 * np.log1p(glow), 17.5, 22.0)
  bortle = np.clip(np.rint(9.2 - (sqm - 17.5) * 1.65), 1, 9).astype(int)
  return glow, sqm, bortle


def build_observation_plan(
    site_name, target_type, moon_phase, max_cloud_pct, min_alt_deg
):
  site_row = next(
      (s for s in CANDIDATE_SITES if s["site"] == site_name),
      CANDIDATE_SITES[0],
  )
  glow_val, sqm_val, bortle_val = sky_glow_at(
      site_row["lat"], site_row["lon"], site_row["elev_m"]
  )
  glow = float(np.asarray(glow_val).item())
  sqm = float(np.asarray(sqm_val).item())
  bortle = int(np.asarray(bortle_val).item())

  azimuths = np.linspace(0, 360, 72, endpoint=False)
  dir_pollution = np.zeros_like(azimuths)
  for src in LIGHT_SOURCES:
    d_km = max(
        float(haversine_km(site_row["lat"], site_row["lon"], src["lat"], src["lon"])),
        1.5,
    )
    b_deg = float(
        bearing_deg(site_row["lat"], site_row["lon"], src["lat"], src["lon"])
    )
    ang_diff = np.degrees(
        np.arctan2(
            np.sin(np.radians(azimuths - b_deg)),
            np.cos(np.radians(azimuths - b_deg)),
        )
    )
    cone_weight = np.exp(-0.5 * (ang_diff / 28.0) ** 2)
    dir_pollution += (src["strength"] * 35.0 / (d_km**1.25)) * cone_weight

  best_az_idx = int(np.argmin(dir_pollution))
  worst_az_idx = int(np.argmax(dir_pollution))
  best_az = float(azimuths[best_az_idx])
  worst_az = float(azimuths[worst_az_idx])

  moon_penalty = (moon_phase / 100.0) * 1.8
  eff_sqm = sqm - moon_penalty
  lim_mag = float(np.clip((eff_sqm - 16.5) * 1.15 + 1.2, 2.0, 6.8))

  target_bortle_cap = {
      "Bright Planets & Moon": 8,
      "Open & Globular Star Clusters": 6,
      "Faint Nebulae & Galaxies (DSO)": 4,
      "Visible Satellite Passes": 7,
  }
  max_allowed = target_bortle_cap.get(target_type, 6)
  suitability = (
      "HIGHLY SUITABLE"
      if bortle <= max_allowed - 1 and moon_phase < 65
      else (
          "MODERATE / CONDITIONAL"
          if bortle <= max_allowed
          else "CHALLENGING — PICK DARKER SITE"
      )
  )

  lat_grid = np.linspace(36.90, 37.90, 90)
  lon_grid = np.linspace(35.20, 36.20, 90)
  LON_M, LAT_M = np.meshgrid(lon_grid, lat_grid)
  _, SQM_M, _ = sky_glow_at(LAT_M, LON_M, elev_m=400.0)

  plt.close("all")
  plt.style.use("dark_background")
  fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13.5, 5.2), dpi=110)
  fig.patch.set_facecolor("#0b0f17")

  cf = ax1.contourf(
      LON_M, LAT_M, SQM_M, levels=18, cmap="inferno", alpha=0.92
  )
  cbar = fig.colorbar(cf, ax=ax1, pad=0.02)
  cbar.set_label(
      "Estimated Sky Brightness (mag/arcsec²)", color="#cbd5e1", fontsize=9
  )
  cbar.ax.tick_params(colors="#cbd5e1", labelsize=8)

  for s in CANDIDATE_SITES:
    is_sel = s["site"] == site_row["site"]
    ax1.scatter(
        s["lon"],
        s["lat"],
        s=130 if is_sel else 65,
        c="#38bdf8" if is_sel else "#e2e8f0",
        edgecolors="#0b0f17",
        linewidths=1.5,
        zorder=5,
    )
    ax1.text(
        s["lon"] + 0.015,
        s["lat"] + 0.012,
        s["site"].split("(")[0].strip(),
        color="#f8fafc",
        fontsize=8,
        fontweight="bold" if is_sel else "normal",
        zorder=6,
    )

  ax1.set_title(
      "Local Light Pollution Map (Kozan & Surroundings)",
      color="#f8fafc",
      fontsize=11,
      fontweight="bold",
  )
  ax1.set_xlabel("Longitude (°E)", color="#94a3b8", fontsize=9)
  ax1.set_ylabel("Latitude (°N)", color="#94a3b8", fontsize=9)
  ax1.tick_params(colors="#94a3b8", labelsize=8)
  ax1.set_facecolor("#111724")

  ax2.plot(
      azimuths, dir_pollution, color="#38bdf8", linewidth=2.2, label="Light Dome"
  )
  ax2.fill_between(
      azimuths, 0, dir_pollution, color="#38bdf8", alpha=0.18
  )
  ax2.axvline(
      best_az,
      color="#34d399",
      linestyle="--",
      linewidth=1.4,
      label=f"Darkest Azimuth ({best_az:.0f}°)",
  )
  ax2.axvline(
      worst_az,
      color="#f87171",
      linestyle=":",
      linewidth=1.4,
      label=f"Brightest Glare ({worst_az:.0f}°)",
  )
  ax2.set_title(
      f"Directional Light Dome — {site_row['site']}",
      color="#f8fafc",
      fontsize=11,
      fontweight="bold",
  )
  ax2.set_xlabel("Azimuth Direction (0°=N, 90°=E, 180°=S, 270°=W)", color="#94a3b8", fontsize=9)
  ax2.set_ylabel("Relative Directional Sky Glow", color="#94a3b8", fontsize=9)
  ax2.set_xlim(0, 360)
  ax2.tick_params(colors="#94a3b8", labelsize=8)
  ax2.grid(True, alpha=0.18, color="#64748b")
  ax2.legend(
      loc="upper right",
      fontsize=8,
      facecolor="#121826",
      edgecolor="#263248",
  )
  ax2.set_facecolor("#111724")

  plt.tight_layout(pad=1.5)

  cards_html = f"""
    <div style="width:100%;box-sizing:border-box;font-family:Inter,system-ui,sans-serif;color:#e2e8f0;">
      <div style="display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:10px;background:#121826;border:1px solid #263248;border-radius:12px;padding:14px 18px;margin-bottom:12px;">
        <div>
          <div style="font-size:11px;color:#38bdf8;font-weight:700;letter-spacing:0.5px;">ASTRONOMY OBSERVATION PLAN • {site_row['elev_m']} m ELEVATION</div>
          <div style="font-size:20px;font-weight:700;color:#f8fafc;margin-top:2px;">{site_row['site']} <span style="font-size:13px;color:#94a3b8;">({site_row['lat']:.3f}°N, {site_row['lon']:.3f}°E)</span></div>
        </div>
        <span style="background:#0f293a;color:#38bdf8;border:1px solid #0284c7;padding:6px 12px;border-radius:999px;font-size:12px;font-weight:700;">Bortle {bortle} • {suitability}</span>
      </div>

      <div style="display:grid;grid-template-columns:repeat(4,minmax(0,1fr));gap:10px;">
        <div style="background:#131a29;border:1px solid #243044;border-radius:10px;padding:12px;">
          <div style="font-size:11px;color:#94a3b8;">BORTLE CLASS</div>
          <div style="font-size:19px;font-weight:700;color:#38bdf8;margin-top:4px;">Class {bortle} / 9</div>
        </div>
        <div style="background:#131a29;border:1px solid #243044;border-radius:10px;padding:12px;">
          <div style="font-size:11px;color:#94a3b8;">ZENITH SQM</div>
          <div style="font-size:19px;font-weight:700;color:#a78bfa;margin-top:4px;">{sqm:.2f} mag/arcsec²</div>
        </div>
        <div style="background:#131a29;border:1px solid #243044;border-radius:10px;padding:12px;">
          <div style="font-size:11px;color:#94a3b8;">BEST AZIMUTH</div>
          <div style="font-size:19px;font-weight:700;color:#34d399;margin-top:4px;">{best_az:.0f}°</div>
        </div>
        <div style="background:#131a29;border:1px solid #243044;border-radius:10px;padding:12px;">
          <div style="font-size:11px;color:#94a3b8;">EST. LIMITING MAG</div>
          <div style="font-size:19px;font-weight:700;color:#fbbf24;margin-top:4px;">{lim_mag:.1f} mag</div>
        </div>
      </div>

      <div style="background:#111724;border:1px solid #263248;border-radius:12px;padding:15px 18px;margin-top:12px;line-height:1.55;font-size:14px;color:#cbd5e1;">
        <div style="font-weight:700;color:#f8fafc;margin-bottom:6px;">🔭 Actionable Observation Plan — {target_type}</div>
        <ul style="margin:0;padding-left:18px;">
          <li><b>Aim Direction:</b> Favor azimuths near <b>{best_az:.0f}°</b> and keep targets above <b>{min_alt_deg:.0f}°</b> altitude; avoid low-elevation pointing toward <b>{worst_az:.0f}°</b> where urban light domes scatter sky background.</li>
          <li><b>Moon & Sky Window:</b> Current Moon illumination is <b>{moon_phase:.0f}%</b> (cloud cap <b>≤ {max_cloud_pct:.0f}%</b>). For deep-sky targets, prioritize moonless nights (&lt; 25% illumination) after astronomical twilight.</li>
          <li><b>Site Strategy:</b> Moving north/upslope away from urban cores reduces lower-atmosphere scattering and improves contrast for faint objects.</li>
        </ul>
      </div>
    </div>
    """

  schedule_df = pd.DataFrame([
      {
          "Session Window (Local)": "20:00 – 21:30 (Evening Twilight)",
          "Recommended Focus": "Moon, Planets, Bright Clusters & LEO Passes",
          "Min Elevation": f"{max(min_alt_deg, 25):.0f}°",
          "Notes": "Quick setup; early sky gradient near urban corridors",
      },
      {
          "Session Window (Local)": "21:30 – 01:30 (Prime Dark Window)",
          "Recommended Focus": target_type,
          "Min Elevation": f"{min_alt_deg:.0f}°",
          "Notes": f"Aim near {best_az:.0f}° az; check cloud cover <= {max_cloud_pct:.0f}%",
      },
      {
          "Session Window (Local)": "01:30 – 04:30 (Late Night / Pre-Dawn)",
          "Recommended Focus": "Deep-Sky & Low-Glare Meridian Targets",
          "Min Elevation": f"{max(min_alt_deg, 30):.0f}°",
          "Notes": "Reduced street/commercial lighting activity in nearby towns",
      },
  ])

  return cards_html, fig, schedule_df


full_width_css = """
html, body, .gradio-container, .gradio-container > .main, .gradio-container > .main > .wrap, .block, .form {
  background-color: #0b0f17 !important;
  color: #e2e8f0 !important;
  max-width: 100% !important;
  width: 100% !important;
  margin: 0 !important;
  box-sizing: border-box !important;
}
.gradio-container {
  padding: 10px 14px !important;
}
"""

with gr.Blocks() as demo:
  gr.Markdown("## 🌌 Local Light Pollution & Astronomy Observation Planner")

  with gr.Row(equal_height=False):
    with gr.Column(scale=4, min_width=320):
      site_dd = gr.Dropdown(
          choices=[s["site"] for s in CANDIDATE_SITES],
          value=CANDIDATE_SITES[1]["site"],
          label="Candidate Observation Site",
      )
      target_dd = gr.Dropdown(
          choices=[
              "Bright Planets & Moon",
              "Open & Globular Star Clusters",
              "Faint Nebulae & Galaxies (DSO)",
              "Visible Satellite Passes",
          ],
          value="Open & Globular Star Clusters",
          label="Observation Target Type",
      )
      moon_sl = gr.Slider(
          minimum=0,
          maximum=100,
          value=15,
          step=1,
          label="Moon Illumination (%)",
      )
      cloud_sl = gr.Slider(
          minimum=0,
          maximum=100,
          value=20,
          step=5,
          label="Max Acceptable Cloud Cover (%)",
      )
      alt_sl = gr.Slider(
          minimum=10,
          maximum=75,
          value=25,
          step=5,
          label="Minimum Target Elevation (deg)",
      )
      plan_btn = gr.Button(
          "⚡ Map Light Pollution & Build Observation Plan", variant="primary"
      )

    with gr.Column(scale=8, min_width=460):
      out_cards = gr.HTML()
      out_plot = gr.Plot(label="Light Pollution Map & Directional Light Dome")
      out_table = gr.Dataframe(
          label="Nightly Observation Schedule", interactive=False
      )

  plan_btn.click(
      fn=build_observation_plan,
      inputs=[site_dd, target_dd, moon_sl, cloud_sl, alt_sl],
      outputs=[out_cards, out_plot, out_table],
  )

demo.launch(share=False, theme=gr.themes.Soft(), css=full_width_css)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

In [3]:
!pip install -q gradio numpy pandas matplotlib

import gradio as gr
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Light sources around Kozan / Adana region (name, lat, lon, light_intensity_index)
LIGHT_SOURCES = [
    {"name": "Kozan Center", "lat": 37.455, "lon": 35.815, "strength": 1.35},
    {"name": "Imamoglu / Kozan South Corridor", "lat": 37.258, "lon": 35.662, "strength": 1.15},
    {"name": "Adana Metro Urban Core", "lat": 37.000, "lon": 35.321, "strength": 4.20},
    {"name": "Ceyhan Industrial / Urban Node", "lat": 37.026, "lon": 35.817, "strength": 2.10},
    {"name": "Kadirli / Osmaniye Corridor", "lat": 37.372, "lon": 36.094, "strength": 1.75},
    {"name": "Feke / Saimbeyli Uplands", "lat": 37.818, "lon": 35.915, "strength": 0.45},
]

CANDIDATE_SITES = [
    {"site": "Kozan Town Outskirts (Lowland)", "lat": 37.445, "lon": 35.845, "elev_m": 220},
    {"site": "Kozan Northern Ridge / Highland", "lat": 37.535, "lon": 35.795, "elev_m": 680},
    {"site": "Tufanbeyli / Toros Foothill Dark Spot", "lat": 37.690, "lon": 35.870, "elev_m": 1150},
    {"site": "Southern Plain Near Highway Corridor", "lat": 37.310, "lon": 35.710, "elev_m": 110},
]


def haversine_km(lat1, lon1, lat2, lon2):
  r = 6371.0
  p1, p2 = np.radians(lat1), np.radians(lat2)
  dp = np.radians(lat2 - lat1)
  dl = np.radians(lon2 - lon1)
  a = np.sin(dp / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dl / 2) ** 2
  return 2 * r * np.arcsin(np.sqrt(np.clip(a, 0.0, 1.0)))


def bearing_deg(lat1, lon1, lat2, lon2):
  p1, p2 = np.radians(lat1), np.radians(lat2)
  dl = np.radians(lon2 - lon1)
  x = np.sin(dl) * np.cos(p2)
  y = np.cos(p1) * np.sin(p2) - np.sin(p1) * np.cos(p2) * np.cos(dl)
  return (np.degrees(np.arctan2(x, y)) + 360.0) % 360.0


def sky_glow_at(lat, lon, elev_m=200.0):
  total_glow = np.zeros_like(np.asarray(lat, dtype=float)) + 0.25
  for src in LIGHT_SOURCES:
    d_km = np.maximum(haversine_km(lat, lon, src["lat"], src["lon"]), 1.5)
    total_glow = total_glow + (src["strength"] * 42.0 / (d_km**1.35))
  elev_factor = np.clip(1.0 - (np.asarray(elev_m, dtype=float) - 150.0) / 3500.0, 0.80, 1.05)
  glow = total_glow * elev_factor
  sqm = np.clip(21.95 - 1.15 * np.log1p(glow), 17.5, 22.0)
  bortle = np.clip(np.rint(9.2 - (sqm - 17.5) * 1.65), 1, 9).astype(int)
  return glow, sqm, bortle


def build_observation_plan(
    site_name, target_type, moon_phase, max_cloud_pct, min_alt_deg
):
  site_row = next(
      (s for s in CANDIDATE_SITES if s["site"] == site_name),
      CANDIDATE_SITES[0],
  )
  glow_val, sqm_val, bortle_val = sky_glow_at(
      site_row["lat"], site_row["lon"], site_row["elev_m"]
  )
  glow = float(np.asarray(glow_val).item())
  sqm = float(np.asarray(sqm_val).item())
  bortle = int(np.asarray(bortle_val).item())

  azimuths = np.linspace(0, 360, 72, endpoint=False)
  dir_pollution = np.zeros_like(azimuths)
  for src in LIGHT_SOURCES:
    d_km = max(
        float(haversine_km(site_row["lat"], site_row["lon"], src["lat"], src["lon"])),
        1.5,
    )
    b_deg = float(
        bearing_deg(site_row["lat"], site_row["lon"], src["lat"], src["lon"])
    )
    ang_diff = np.degrees(
        np.arctan2(
            np.sin(np.radians(azimuths - b_deg)),
            np.cos(np.radians(azimuths - b_deg)),
        )
    )
    cone_weight = np.exp(-0.5 * (ang_diff / 28.0) ** 2)
    dir_pollution += (src["strength"] * 35.0 / (d_km**1.25)) * cone_weight

  best_az_idx = int(np.argmin(dir_pollution))
  worst_az_idx = int(np.argmax(dir_pollution))
  best_az = float(azimuths[best_az_idx])
  worst_az = float(azimuths[worst_az_idx])

  moon_penalty = (moon_phase / 100.0) * 1.8
  eff_sqm = sqm - moon_penalty
  lim_mag = float(np.clip((eff_sqm - 16.5) * 1.15 + 1.2, 2.0, 6.8))

  target_bortle_cap = {
      "Bright Planets & Moon": 8,
      "Open & Globular Star Clusters": 6,
      "Faint Nebulae & Galaxies (DSO)": 4,
      "Visible Satellite Passes": 7,
  }
  max_allowed = target_bortle_cap.get(target_type, 6)
  suitability = (
      "HIGHLY SUITABLE"
      if bortle <= max_allowed - 1 and moon_phase < 65
      else (
          "MODERATE / CONDITIONAL"
          if bortle <= max_allowed
          else "CHALLENGING — PICK DARKER SITE"
      )
  )

  lat_grid = np.linspace(36.90, 37.90, 90)
  lon_grid = np.linspace(35.20, 36.20, 90)
  LON_M, LAT_M = np.meshgrid(lon_grid, lat_grid)
  _, SQM_M, _ = sky_glow_at(LAT_M, LON_M, elev_m=400.0)

  plt.close("all")
  plt.style.use("dark_background")
  fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13.5, 5.0), dpi=110)
  fig.patch.set_facecolor("#0b0f17")

  cf = ax1.contourf(
      LON_M, LAT_M, SQM_M, levels=18, cmap="inferno", alpha=0.92
  )
  cbar = fig.colorbar(cf, ax=ax1, pad=0.02)
  cbar.set_label(
      "Estimated Sky Brightness (mag/arcsec²)", color="#cbd5e1", fontsize=9
  )
  cbar.ax.tick_params(colors="#cbd5e1", labelsize=8)

  for s in CANDIDATE_SITES:
    is_sel = s["site"] == site_row["site"]
    ax1.scatter(
        s["lon"],
        s["lat"],
        s=130 if is_sel else 65,
        c="#38bdf8" if is_sel else "#e2e8f0",
        edgecolors="#0b0f17",
        linewidths=1.5,
        zorder=5,
    )
    ax1.text(
        s["lon"] + 0.015,
        s["lat"] + 0.012,
        s["site"].split("(")[0].strip(),
        color="#f8fafc",
        fontsize=8,
        fontweight="bold" if is_sel else "normal",
        zorder=6,
    )

  ax1.set_title(
      "Local Light Pollution Map (Kozan & Surroundings)",
      color="#f8fafc",
      fontsize=11,
      fontweight="bold",
  )
  ax1.set_xlabel("Longitude (°E)", color="#94a3b8", fontsize=9)
  ax1.set_ylabel("Latitude (°N)", color="#94a3b8", fontsize=9)
  ax1.tick_params(colors="#94a3b8", labelsize=8)
  ax1.set_facecolor("#111724")

  ax2.plot(
      azimuths, dir_pollution, color="#38bdf8", linewidth=2.2, label="Light Dome"
  )
  ax2.fill_between(
      azimuths, 0, dir_pollution, color="#38bdf8", alpha=0.18
  )
  ax2.axvline(
      best_az,
      color="#34d399",
      linestyle="--",
      linewidth=1.4,
      label=f"Darkest Azimuth ({best_az:.0f}°)",
  )
  ax2.axvline(
      worst_az,
      color="#f87171",
      linestyle=":",
      linewidth=1.4,
      label=f"Brightest Glare ({worst_az:.0f}°)",
  )
  ax2.set_title(
      f"Directional Light Dome — {site_row['site']}",
      color="#f8fafc",
      fontsize=11,
      fontweight="bold",
  )
  ax2.set_xlabel("Azimuth Direction (0°=N, 90°=E, 180°=S, 270°=W)", color="#94a3b8", fontsize=9)
  ax2.set_ylabel("Relative Directional Sky Glow", color="#94a3b8", fontsize=9)
  ax2.set_xlim(0, 360)
  ax2.tick_params(colors="#94a3b8", labelsize=8)
  ax2.grid(True, alpha=0.18, color="#64748b")
  ax2.legend(
      loc="upper right",
      fontsize=8,
      facecolor="#121826",
      edgecolor="#263248",
  )
  ax2.set_facecolor("#111724")

  plt.tight_layout(pad=1.5)

  cards_html = f"""
    <div style="width:100%;box-sizing:border-box;font-family:Inter,system-ui,sans-serif;color:#e2e8f0;">
      <div style="display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:10px;background:#121826;border:1px solid #263248;border-radius:12px;padding:14px 18px;margin-bottom:12px;">
        <div>
          <div style="font-size:11px;color:#38bdf8;font-weight:700;letter-spacing:0.5px;">ASTRONOMY OBSERVATION PLAN • {site_row['elev_m']} m ELEVATION</div>
          <div style="font-size:20px;font-weight:700;color:#f8fafc;margin-top:2px;">{site_row['site']} <span style="font-size:13px;color:#94a3b8;">({site_row['lat']:.3f}°N, {site_row['lon']:.3f}°E)</span></div>
        </div>
        <span style="background:#0f293a;color:#38bdf8;border:1px solid #0284c7;padding:6px 12px;border-radius:999px;font-size:12px;font-weight:700;">Bortle {bortle} • {suitability}</span>
      </div>

      <div style="display:grid;grid-template-columns:repeat(4,minmax(0,1fr));gap:10px;">
        <div style="background:#131a29;border:1px solid #243044;border-radius:10px;padding:12px;">
          <div style="font-size:11px;color:#94a3b8;">BORTLE CLASS</div>
          <div style="font-size:19px;font-weight:700;color:#38bdf8;margin-top:4px;">Class {bortle} / 9</div>
        </div>
        <div style="background:#131a29;border:1px solid #243044;border-radius:10px;padding:12px;">
          <div style="font-size:11px;color:#94a3b8;">ZENITH SQM</div>
          <div style="font-size:19px;font-weight:700;color:#a78bfa;margin-top:4px;">{sqm:.2f} mag/arcsec²</div>
        </div>
        <div style="background:#131a29;border:1px solid #243044;border-radius:10px;padding:12px;">
          <div style="font-size:11px;color:#94a3b8;">BEST AZIMUTH</div>
          <div style="font-size:19px;font-weight:700;color:#34d399;margin-top:4px;">{best_az:.0f}°</div>
        </div>
        <div style="background:#131a29;border:1px solid #243044;border-radius:10px;padding:12px;">
          <div style="font-size:11px;color:#94a3b8;">EST. LIMITING MAG</div>
          <div style="font-size:19px;font-weight:700;color:#fbbf24;margin-top:4px;">{lim_mag:.1f} mag</div>
        </div>
      </div>
    </div>
    """

  plan_box_html = f"""
    <div style="width:100%;box-sizing:border-box;font-family:Inter,system-ui,sans-serif;background:#111724;border:1px solid #263248;border-radius:12px;padding:16px 20px;line-height:1.6;font-size:14px;color:#cbd5e1;">
      <div style="font-weight:700;font-size:15px;color:#f8fafc;margin-bottom:8px;">🔭 Actionable Observation Plan — {target_type} ({site_row['site']})</div>
      <ul style="margin:0;padding-left:20px;">
        <li><b>Aim Direction:</b> Favor azimuths near <b>{best_az:.0f}°</b> and keep targets above <b>{min_alt_deg:.0f}°</b> altitude; avoid low-elevation pointing toward <b>{worst_az:.0f}°</b> where urban light domes scatter sky background.</li>
        <li><b>Moon & Sky Window:</b> Current Moon illumination is <b>{moon_phase:.0f}%</b> (cloud cap <b>≤ {max_cloud_pct:.0f}%</b>). For deep-sky targets, prioritize moonless nights (&lt; 25% illumination) after astronomical twilight.</li>
        <li><b>Site Strategy:</b> Moving north/upslope away from urban cores reduces lower-atmosphere scattering and improves contrast for faint objects.</li>
      </ul>
    </div>
    """

  schedule_df = pd.DataFrame([
      {
          "Session Window (Local)": "20:00 – 21:30 (Evening Twilight)",
          "Recommended Focus": "Moon, Planets, Bright Clusters & LEO Passes",
          "Min Elevation": f"{max(min_alt_deg, 25):.0f}°",
          "Notes": "Quick setup; early sky gradient near urban corridors",
      },
      {
          "Session Window (Local)": "21:30 – 01:30 (Prime Dark Window)",
          "Recommended Focus": target_type,
          "Min Elevation": f"{min_alt_deg:.0f}°",
          "Notes": f"Aim near {best_az:.0f}° az; check cloud cover <= {max_cloud_pct:.0f}%",
      },
      {
          "Session Window (Local)": "01:30 – 04:30 (Late Night / Pre-Dawn)",
          "Recommended Focus": "Deep-Sky & Low-Glare Meridian Targets",
          "Min Elevation": f"{max(min_alt_deg, 30):.0f}°",
          "Notes": "Reduced street/commercial lighting activity in nearby towns",
      },
  ])

  return cards_html, fig, plan_box_html, schedule_df


full_width_css = """
html, body, .gradio-container, .gradio-container > .main, .gradio-container > .main > .wrap, .block, .form {
  background-color: #0b0f17 !important;
  color: #e2e8f0 !important;
  max-width: 100% !important;
  width: 100% !important;
  margin: 0 !important;
  box-sizing: border-box !important;
}
.gradio-container {
  padding: 10px 14px !important;
}
.table-wrap, .dataframe {
  width: 100% !important;
  max-width: 100% !important;
  overflow-x: visible !important;
}
"""

with gr.Blocks() as demo:
  gr.Markdown("## 🌌 Local Light Pollution & Astronomy Observation Planner")

  with gr.Row(equal_height=False):
    with gr.Column(scale=4, min_width=320):
      site_dd = gr.Dropdown(
          choices=[s["site"] for s in CANDIDATE_SITES],
          value=CANDIDATE_SITES[1]["site"],
          label="Candidate Observation Site",
      )
      target_dd = gr.Dropdown(
          choices=[
              "Bright Planets & Moon",
              "Open & Globular Star Clusters",
              "Faint Nebulae & Galaxies (DSO)",
              "Visible Satellite Passes",
          ],
          value="Open & Globular Star Clusters",
          label="Observation Target Type",
      )
      moon_sl = gr.Slider(
          minimum=0,
          maximum=100,
          value=15,
          step=1,
          label="Moon Illumination (%)",
      )
      cloud_sl = gr.Slider(
          minimum=0,
          maximum=100,
          value=20,
          step=5,
          label="Max Acceptable Cloud Cover (%)",
      )
      alt_sl = gr.Slider(
          minimum=10,
          maximum=75,
          value=25,
          step=5,
          label="Minimum Target Elevation (deg)",
      )
      plan_btn = gr.Button(
          "⚡ Map Light Pollution & Build Observation Plan", variant="primary"
      )

    with gr.Column(scale=8, min_width=460):
      out_cards = gr.HTML()
      out_plot = gr.Plot(label="Light Pollution Map & Directional Light Dome")

  with gr.Row():
    with gr.Column(scale=12):
      out_plan_box = gr.HTML()
      out_table = gr.Dataframe(
          label="Nightly Observation Schedule", interactive=False
      )

  plan_btn.click(
      fn=build_observation_plan,
      inputs=[site_dd, target_dd, moon_sl, cloud_sl, alt_sl],
      outputs=[out_cards, out_plot, out_plan_box, out_table],
  )

demo.launch(share=False, theme=gr.themes.Soft(), css=full_width_css)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>